# Aula 21 — Preparação de texto e expressões regulares

Uma notícia parece pronta para leitura humana. Para comparar centenas delas, precisamos decidir o que será contado como palavra e o que é ruído. Cada corte muda o que conseguimos enxergar. Hoje vamos registrar essas decisões, sempre olhando o texto antes e depois.

**Roteiro:** carregar → inspecionar → aprender regex → preparar → lematizar → conferir perdas → salvar. Usaremos 500 notícias em português brasileiro de cinco editorias. A [fonte está no GitHub](https://github.com/mauricio-seiji/Dataset-news-articles-pt-br); o notebook baixa os textos durante a execução. Abra esta aula no Google Colab e rode as células na ordem.

## 1. Relembrando a tabela

Cada linha será uma notícia. `editoria` é um rótulo fornecido pelo corpus, não uma conclusão nossa. Antes de limpar, confira tamanho, vazios, duplicatas e um texto inteiro. Nesta aula não precisamos de GPU de GPU; a seção de lematização instala o spaCy e o modelo de português no Colab.

In [ ]:
import io
import zipfile
import requests
import pandas as pd

url = "https://codeload.github.com/mauricio-seiji/Dataset-news-articles-pt-br/zip/refs/heads/main"
resposta = requests.get(url, timeout=40)
resposta.raise_for_status()
zip_fonte = zipfile.ZipFile(io.BytesIO(resposta.content))
linhas = []
for caminho in zip_fonte.namelist():
    if caminho.endswith(".txt") and "__summarization__" not in caminho:
        partes = caminho.split("/")
        linhas.append({"id": partes[-2] + "/" + partes[-1],
                       "editoria": partes[-2],
                       "texto_original": zip_fonte.read(caminho).decode("utf-8")})
noticias = pd.DataFrame(linhas).sort_values("id").reset_index(drop=True)
print(noticias.shape)
print(noticias["editoria"].value_counts())
assert len(noticias) == 500

In [ ]:
print("Vazios:", noticias["texto_original"].str.strip().eq("").sum())
print("Duplicatas exatas:", noticias["texto_original"].duplicated().sum())
print("Mediana de caracteres:", noticias["texto_original"].str.len().median())
print(noticias.loc[0, "texto_original"][:700])

**O que observar:** nenhum texto está vazio; há uma duplicata exata. O corpus também contém notícias curtas e longas e ao menos uma notícia cuja editoria parece estranha. Uma base real pede inspeção antes de interpretação.

## 2. Regex: localizar e extrair padrões

Uma expressão regular descreve uma *forma* de texto. Primeiro fazemos uma pergunta simples com `re.search()` (há ocorrência?). Depois usamos `re.findall()` (quais são?) e `re.sub()` (como substituir?). Prefixar o padrão com `r` evita que o Python interprete a barra invertida antes da regex.

Neste exemplo controlado sabemos o resultado esperado. `\b` marca limite de palavra; `\d` representa um dígito; `{1,2}` exige um ou dois; `?` torna o caractere anterior opcional; `+` exige uma ou mais repetições. Não precisamos decorar tudo: testamos cada padrão e conferimos o que ele capturou.

In [ ]:
import re
exemplo = "Em 07/10/2026, NASA publicou dados em https://exemplo.org/a. Em 8/10/2026, a nasa respondeu."
data = r"\b\d{1,2}/\d{1,2}/\d{4}\b"
url = r"https?://\S+"
print("Há data?", bool(re.search(data, exemplo)))
print("Datas:", re.findall(data, exemplo))
print("URLs:", re.findall(url, exemplo))
print("NASA (sem diferenciar caixa):", re.findall(r"\bnasa\b", exemplo, flags=re.IGNORECASE))
print("Sem URL:", re.sub(url, "[URL]", exemplo))
assert re.findall(data, exemplo) == ["07/10/2026", "8/10/2026"]
assert len(re.findall(r"\bnasa\b", exemplo, flags=re.IGNORECASE)) == 2

Agora tente nos textos reais. Uma notícia pode não ter nenhuma data no formato procurado: lista vazia é um resultado válido. Note que a URL do exemplo controlado veio com o ponto final: `\S+` inclui pontuação ao fim da frase. A regex encontra **formatos**, mas não verifica se uma data existe no calendário nem se um número é mesmo um valor monetário. Antes de apagar padrões, inspecione exemplos.

In [ ]:
amostra = noticias.loc[0, "texto_original"]
for nome, padrao in {"datas": data, "URLs": url, "R$": r"R\$\s?\d+(?:[.,]\d+)*"}.items():
    achados = re.findall(padrao, amostra, flags=re.IGNORECASE)
    print(nome, "—", len(achados), "ocorrências; exemplos:", achados[:3])
print("Trecho original:", amostra[:300])

**Teste rápido:** troque `\bnasa\b` por `nasa` no exemplo controlado. Por que uma palavra como `nasal` passaria a aparecer? Depois mude `https?` para `https`: qual URL deixaria de ser aceita? Essas são escolhas de regra, não detalhes de sintaxe.

## 3. Tokenizar sem apagar informação importante

Token é uma unidade de texto; aqui será uma sequência de letras, incluindo acentos. Lowercase ajuda a juntar `Brasil` e `brasil`, mas pode esconder nomes próprios. Remover acentos pode ajudar uma busca, porém também altera a forma original. Por isso manteremos duas colunas.

In [ ]:
import unicodedata

def sem_acentos(texto):
    partes = unicodedata.normalize("NFD", texto)
    return "".join(letra for letra in partes if unicodedata.category(letra) != "Mn")

def preparar(texto, remover_acentos=False, remover_urls=True):
    texto = str(texto).lower()
    if remover_urls:
        texto = re.sub(r"https?://\S+", " ", texto)
    if remover_acentos:
        texto = sem_acentos(texto)
    tokens = re.findall(r"\b[^\W\d_]+\b", texto, flags=re.UNICODE)
    return " ".join(tokens)

noticias["texto_preparado"] = noticias["texto_original"].map(preparar)
noticias[["id", "editoria", "texto_preparado"]].head(3)

**Ponto de conferência:** a coluna original continua intacta. Veja a primeira notícia lado a lado. O que aconteceria com `São Paulo` se removêssemos acentos? E com uma URL que identifica a fonte?

In [ ]:
for i in [0, 1, 2]:
    print("ID:", noticias.loc[i, "id"])
    print("ANTES:", noticias.loc[i, "texto_original"][:180])
    print("DEPOIS:", noticias.loc[i, "texto_preparado"][:180], "\n")

## 4. Stopwords e lematização

Stopwords são palavras frequentes, como artigos e preposições. Removê-las pode ajudar um modelo de tópicos, mas também pode apagar negação: `não apoiou` viraria `apoiou`. Testaremos uma lista pequena e visível, sem `não`.

**Stemming** corta finais de palavras por regras; **lematização** usa uma análise linguística para buscar a forma de dicionário. O spaCy nos permite comparar isso no português. Lema não é correção automática: pode alterar nomes ou sentidos, então guardaremos três versões — original, preparada por regex e lematizada — e auditaremos exemplos.

In [ ]:
stopwords_pequenas = {"a", "o", "as", "os", "de", "da", "do", "das", "dos", "em", "um", "uma", "que", "e"}
frase = "O candidato não apoiou a proposta da escola"
print("Original:", frase)
print("Preparada:", preparar(frase))
print("Sem stopwords:", " ".join(t for t in preparar(frase).split() if t not in stopwords_pequenas))
print("Sem acentos:", preparar(frase, remover_acentos=True))

### Lematização com spaCy

No Colab, a célula abaixo instala o spaCy e o modelo `pt_core_news_sm` apenas se necessário. O modelo é baixado uma vez por ambiente. Usaremos CPU. Primeiro observe uma frase curta; depois processaremos as 500 notícias com `nlp.pipe()` e desativaremos análise sintática e reconhecimento de entidades, que não são necessários para obter lemas.

In [ ]:
import importlib.util
import subprocess
import sys

modelo_url = "https://github.com/explosion/spacy-models/releases/download/pt_core_news_sm-3.8.0/pt_core_news_sm-3.8.0-py3-none-any.whl"
if importlib.util.find_spec("spacy") is None or importlib.util.find_spec("pt_core_news_sm") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy>=3.8,<3.9", modelo_url], check=True)
import spacy
nlp = spacy.load("pt_core_news_sm", disable=["parser", "ner"])

frase = "As pesquisadoras analisaram os dados e publicaram os resultados."
print([(token.text, token.lemma_) for token in nlp(frase) if token.is_alpha])

In [ ]:
def lemas_do_documento(doc):
    return " ".join(token.lemma_.lower() for token in doc if token.is_alpha)

noticias["texto_lematizado"] = [
    lemas_do_documento(doc)
    for doc in nlp.pipe(noticias["texto_original"].astype(str), batch_size=16)
]
for i in [0, 1, 2]:
    print("ID:", noticias.loc[i, "id"])
    print("REGEX:", noticias.loc[i, "texto_preparado"][:180])
    print("LEMAS:", noticias.loc[i, "texto_lematizado"][:180], "\n")
print("Textos lematizados vazios:", noticias["texto_lematizado"].str.strip().eq("").sum())

**Para discutir:** compare um verbo flexionado e um nome próprio nas três versões. A lematização aproximou formas que você gostaria de contar juntas? Mudou alguma palavra que deveria permanecer como estava? Na Aula 22, começaremos com a versão simples por regex e poderemos testar se os lemas mudam os tópicos.

## 5. Medir o efeito da limpeza

Quantas palavras sobraram? Quais palavras frequentes fazem sentido? Compare uma matéria de política e outra de esporte. Não interprete uma lista de frequências como tema do corpus inteiro.

In [ ]:
from collections import Counter
noticias["tokens_antes"] = noticias["texto_original"].str.split().str.len()
noticias["tokens_depois"] = noticias["texto_preparado"].str.split().str.len()
print(noticias[["tokens_antes", "tokens_depois"]].describe().round(1))
for editoria in ["politica", "esportes"]:
    texto = " ".join(noticias.loc[noticias.editoria.eq(editoria), "texto_preparado"])
    frequencias = Counter(t for t in texto.split() if t not in stopwords_pequenas)
    print(editoria, frequencias.most_common(12))

## 6. Quando der errado

- **`HTTPError` ou falta de internet:** o arquivo do corpus não chegou; reconecte o Colab e execute a célula de download novamente.
- **Regex não encontra nada:** imprima um pedaço do texto e teste o padrão em uma frase curta.
- **Tudo vira uma palavra só:** confira se você usou `" ".join(tokens)` e se a regex encontrou tokens.
- **Modelo do spaCy não carrega:** execute novamente a célula de instalação e confira se o runtime tem internet.
- **Palavras importantes sumiram:** volte ao original e mude apenas uma regra por vez.

**Missão final:** abra o exercício. Seu pipeline deve preservar o original, explicar as escolhas e mostrar pelo menos um caso em que limpar demais prejudicaria a análise da aula seguinte.